<font size=10>**PREQUENTIAL EVALUATION**</font> <a class="anchor" id='title'></a>

**Bachelor's in Data Science - NOVA IMS (26/27)**

**Course**: *Data Stream Mining — Practical Assignment I (PA48)*

**Project**: *Weather Windows for Outdoor Races — Angra do Heroísmo vs Braga*

**Group YY**
- Beatriz Marques 20231605
- Duarte Fernandes 20231619

**Question**: *Which city offers more reliable weather windows for recurring outdoor races?*

**Pipeline step**: This notebook reproduces **step 3 and saving** of `python src/practical_assignment.py --config config.yaml`: rebuild the streams, run warm-up and test-then-train evaluation for every city, representation and learner, and write the three audit outputs. The single-run examples are separate demonstrations with fresh models.

| Notebook | What it does |
|---|---|
| 01 — `01_data_collection_and_quality.ipynb` | Data collection and quality |
| 02 — `02_target_and_features.ipynb` | Target, features and shared mask |
| **03 — `03_prequential_evaluation.ipynb`** | **Evaluation and audit outputs** |
| 04 — `04_results_and_figures.ipynb` | Results, figures and audit checks |

<font color='#BFD72F' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>

- [1. Imports](#1)

- [2. Experiment Setup](#2)
    - [2.1 Configuration & Seed](#2_1)
    - [2.2 Streams](#2_2)

- [3. Inside One Run](#3)
    - [3.1 Predict → Evaluate → Learn by Hand](#3_1)
    - [3.2 One Full Run with run_prequential](#3_2)
    - [3.3 Model Complexity](#3_3)
    - [3.4 Cumulative vs Window Macro-F1](#3_4)
        - [Insights](#insights_1)

- [4. Full Experiment](#4)
    - [4.1 Run All Cities × Representations × Models](#4_1)
    - [4.2 Save Audit Outputs](#4_2)
    - [4.3 Quick Look](#4_3)
        - [Insights](#insights_2)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

Enable **automatic source reloads** and suppress warnings while running the notebook cells in order. Start Jupyter with this notebook in the repository’s `notebooks/` folder.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

Import the libraries and the documented **`src` API**. Add `../src` to `sys.path`; subsequent data and output paths are built from **`data_loader.ROOT`**, not the notebook working directory.

In [ ]:
import os
import sys
import random
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import river
import sklearn
import matplotlib
from IPython.display import display

# Get the absolute path of the source code folder
source_code_path = os.path.abspath('../src')

# Add the source code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from data_loader import ROOT, load_config, fetch_city, quality_report
from targets import RAIN_MM, compute_thresholds, make_target
from features import BASE, representation_a, representation_b
from models import build_models, build_baselines, model_complexity
from prequential import ConfusionMatrix, StreamingEvaluator, run_prequential
from plots import generate_plots
%matplotlib inline

# <font color='#BFD72F' size=6>**2. Experiment Setup**</font> <a class="anchor" id="2"></a>

[Back to TOC](#toc)

## <font color='#BFD72F' size=5>2.1 Configuration & Seed</font> <a class="anchor" id="2_1"></a>

[Back to TOC](#toc)

Load **`config.yaml`** and initialise `random` and NumPy with **`cfg["seed"]`**. Read the target, warm-up, rolling evaluation window, memory-tracking flag and baseline switch from their documented configuration keys. All output paths are rooted at `ROOT`.

In [ ]:
cfg = load_config(ROOT / 'config.yaml')
period = cfg['period']
warm_n = cfg['warmup']['n_hours']
raw_dir = ROOT / cfg['paths']['raw_data']
seed = cfg['seed']
random.seed(seed)
np.random.seed(seed)
target = cfg['assignment']['target']
window_size = cfg['evaluation']['window_size']
track_memory = cfg['evaluation']['track_memory']
include_baselines = cfg['baselines']['include']
out = ROOT / cfg['paths']['results']
assert target == 'T6', 'This notebook is designed for the configured T6 assignment.'
display(pd.Series({
    'target': target, 'seed': seed, 'warmup_hours': warm_n,
    'window_size': window_size, 'track_memory': track_memory,
    'include_baselines': include_baselines, 'results_directory': str(out),
}, name='value'))

## <font color='#BFD72F' size=5>2.2 Streams</font> <a class="anchor" id="2_2"></a>

[Back to TOC](#toc)

Rebuild the same **data, frozen threshold, targets, representations and shared masks** explained in notebook **02**. Store each city in `streams[key]`; do not depend on variables from a different notebook kernel.

In [ ]:
data = {}
names = {}
for key in ('city_a', 'city_b'):
    c = cfg['cities'][key]
    names[key] = c['name']
    data[key] = fetch_city(
        c['name'], c['latitude'], c['longitude'],
        period['start_date'], period['end_date'], period['timezone'],
        raw_dir=raw_dir,
    )
thresholds = compute_thresholds(data['city_a'], data['city_b'], warm_n, target)
streams = {}
for key, df in data.items():
    city = cfg['cities'][key]['name']
    y = make_target(df, target, thresholds)
    reps = {
        'A': representation_a(df),
        'B': representation_b(df, cfg['features']['windows'], y=y),
    }
    valid = reps['B'].notna().all(axis=1) & y.notna()
    is_warm = pd.Series(df.index < warm_n, index=df.index)
    streams[key] = dict(city=city, df=df, y=y, reps=reps, valid=valid, is_warm=is_warm)
display(pd.DataFrame([
    {'city': s['city'], 'valid_rows': int(s['valid'].sum()),
     'warmup_rows': int((s['valid'] & s['is_warm']).sum()),
     'evaluation_rows': int((s['valid'] & ~s['is_warm']).sum())}
    for s in streams.values()
]))

# <font color='#BFD72F' size=6>**3. Inside One Run**</font> <a class="anchor" id="3"></a>

[Back to TOC](#toc)

## <font color='#BFD72F' size=5>3.1 Predict → Evaluate → Learn by Hand</font> <a class="anchor" id="3_1"></a>

[Back to TOC](#toc)

Take **Angra, representation B**, and apply the shared mask within the first **`warm_n + 10` original hourly rows**. This preserves the terminal warm-up boundary despite the initial rolling-feature gaps. A fresh **`build_models(seed)["GaussianNB"]`** learns on eligible warm-up rows without scoring; each of the next ten eligible hourly events follows **predict → evaluate → learn**. As in `run_prequential`, a `None` prediction is treated as `0`. `StreamingEvaluator(window_size)` supplies cumulative macro-F1. The check below requires these ten post-warm-up rows to be eligible.

In [ ]:
s = streams['city_a']
manual_model = build_models(seed)['GaussianNB']
manual_evaluator = StreamingEvaluator(window_size)
demo_mask = s['valid'] & pd.Series(s['df'].index < warm_n + 10, index=s['df'].index)
demo_indices = s['df'].index[demo_mask]
assert int((demo_mask & ~s['is_warm']).sum()) == 10, (
    'The first ten post-warm-up hours must be valid for this ten-event demonstration.'
)
manual_rows = []
for idx in demo_indices:
    x_i = s['reps']['B'].loc[idx].to_dict()
    y_i = int(s['y'].loc[idx])
    if bool(s['is_warm'].loc[idx]):
        manual_model.learn_one(x_i, y_i)
        continue
    y_pred = manual_model.predict_one(x_i)
    y_pred = 0 if y_pred is None else int(y_pred)
    manual_evaluator.update(y_i, y_pred)
    manual_rows.append({
        'timestamp': s['df'].loc[idx, 'time'], 'y_true': y_i, 'y_pred': y_pred,
        'cumulative_macro_f1': manual_evaluator.cumulative.macro_f1(),
    })
    manual_model.learn_one(x_i, y_i)
manual_df = pd.DataFrame(manual_rows)
assert len(manual_df) == 10
print(manual_df.to_string(index=False))

## <font color='#BFD72F' size=5>3.2 One Full Run with run_prequential</font> <a class="anchor" id="3_2"></a>

[Back to TOC](#toc)

Create a fresh **`HoeffdingTree`** and run the full **Angra B** stream with `run_prequential`. Pass the same filtered `X`, integer target, original warm-up flag and timestamps as the terminal, plus **`cfg["evaluation"]`** settings.

In [ ]:
s = streams['city_a']
model = build_models(seed)['HoeffdingTree']
summary, history = run_prequential(
    model, s['reps']['B'][s['valid']], s['y'][s['valid']].astype(int),
    s['is_warm'][s['valid']], online=True, window_size=window_size,
    timestamps=s['df'].loc[s['valid'], 'time'].tolist(), track_memory=track_memory,
)
display(pd.Series(summary, name='value'))
display(history.head())

## <font color='#BFD72F' size=5>3.3 Model Complexity</font> <a class="anchor" id="3_3"></a>

[Back to TOC](#toc)

Inspect the **trained example model**, not a new instance, with `model_complexity(model)`. The returned fields cover ensemble size, tree structure and drift/warning counts.

In [ ]:
display(pd.Series(model_complexity(model), name='value'))

## <font color='#BFD72F' size=5>3.4 Cumulative vs Window Macro-F1</font> <a class="anchor" id="3_4"></a>

[Back to TOC](#toc)

Display the example run’s **cumulative** and **rolling-window macro-F1**, using every 24th history row for plotting only. The saved and evaluated history remains unsampled; the rolling window is **`cfg["evaluation"]["window_size"]`** events.

In [ ]:
plot_history = history.iloc[::24]
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(plot_history['timestamp'], plot_history['cumulative_macro_f1'], label='Cumulative macro-F1')
ax.plot(plot_history['timestamp'], plot_history['window_macro_f1'], label=f'Window macro-F1 ({window_size} events)')
ax.set_title(f"{s['city']} | Rep B | HoeffdingTree")
ax.set_xlabel('Timestamp')
ax.set_ylabel('Macro-F1')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

#### **Insights** <a class="anchor" id='insights_1'></a>
*To be written by the group.*

# <font color='#BFD72F' size=6>**4. Full Experiment**</font> <a class="anchor" id="4"></a>

[Back to TOC](#toc)

**Runtime warning:** the following is the same cities × representations × models loop as the terminal command and may take **several minutes**; **ARF is the slowest** learner. Keep `RUN_FULL_EXPERIMENT = True` to reproduce the experiment. Set it to `False` to skip training and load existing audit outputs from the configured results directory. Before the full loop, reset both seeds so the demonstration runs do not change the full experiment’s random state.

In [ ]:
RUN_FULL_EXPERIMENT = True

## <font color='#BFD72F' size=5>4.1 Run All Cities × Representations × Models</font> <a class="anchor" id="4_1"></a>

[Back to TOC](#toc)

Build **fresh mandatory models and optional baselines for every city/representation pair** using `build_models(seed)` and `build_baselines(seed)`. Reuse **B’s `valid` mask** for A and B. Pass each baseline’s `online` flag unchanged so **`FrozenLogReg`** remains frozen after warm-up. Append the evaluator summary and the trained model’s complexity to every metrics row.

In [ ]:
if RUN_FULL_EXPERIMENT:
    random.seed(seed)
    np.random.seed(seed)
    rows = []
    histories = []
    for key, s in streams.items():
        city = s['city']
        df = s['df']
        y = s['y']
        reps = s['reps']
        valid = s['valid']
        is_warm = s['is_warm']
        for rep_name, X in reps.items():
            learners = [(n, m, True, 'mandatory') for n, m in build_models(seed).items()]
            if include_baselines:
                learners += [
                    (n, m, online, 'baseline')
                    for n, (m, online) in build_baselines(seed).items()
                ]
            for model_name, model, online, role in learners:
                res, run_history = run_prequential(
                    model, X[valid], y[valid].astype(int), is_warm[valid],
                    online=online, window_size=window_size,
                    timestamps=df.loc[valid, 'time'].tolist(), track_memory=track_memory,
                )
                rows.append({
                    'city': city, 'representation': rep_name, 'model': model_name,
                    'role': role, **res, **model_complexity(model),
                })
                histories.append(run_history.assign(
                    city=city, representation=rep_name, model=model_name, role=role,
                ))
                print(
                    f"{city} | Rep {rep_name} | {model_name}: "
                    f"MacroF1={res['macro_f1']:.4f} Recall={res['recall']:.4f}"
                )
    metrics_df = pd.DataFrame(rows)
    history_df = pd.concat(histories, ignore_index=True)
else:
    print('Full experiment skipped; existing audit files will be loaded in section 4.2.')

## <font color='#BFD72F' size=5>4.2 Save Audit Outputs</font> <a class="anchor" id="4_2"></a>

[Back to TOC](#toc)

When training is enabled, write **`metrics.csv`**, **`history.csv.gz`** and **`configurations.csv`** to **`ROOT / cfg["paths"]["results"]`**, using the terminal’s schema, gzip compression and version/model audit entries. Baseline descriptions are recorded even when baselines are excluded, as in `main()`. When training is disabled, do not overwrite files; load existing metrics and history instead. Figure generation is handled in notebook **04**.

In [ ]:
if RUN_FULL_EXPERIMENT:
    out.mkdir(parents=True, exist_ok=True)
    metrics_df.to_csv(out / 'metrics.csv', index=False)
    history_df.to_csv(out / 'history.csv.gz', index=False, compression='gzip')
    config_rows = {
        'assignment_code': cfg['assignment']['code'],
        'target': target,
        'city_a': cfg['cities']['city_a']['name'],
        'city_b': cfg['cities']['city_b']['name'],
        'start_date': period['start_date'],
        'end_date': period['end_date'],
        'warmup_hours': warm_n,
        'seed': seed,
        'thresholds': thresholds,
        'window_size': window_size,
        'track_memory': track_memory,
        'include_baselines': include_baselines,
        'python': platform.python_version(),
        'river': river.__version__,
        'scikit-learn': sklearn.__version__,
        'matplotlib': matplotlib.__version__,
        'pandas': pd.__version__,
        'numpy': np.__version__,
        **{f'model_{k}': repr(v) for k, v in build_models(seed).items()},
        **{
            f'baseline_{k}': f'{repr(v)}; online={online}'
            for k, (v, online) in build_baselines(seed).items()
        },
    }
    pd.Series(config_rows).to_csv(out / 'configurations.csv', header=['value'])
    print(f'Audit outputs saved to {out}')
else:
    for filename in ('metrics.csv', 'history.csv.gz', 'configurations.csv'):
        if not (out / filename).exists():
            raise FileNotFoundError(f'Missing {out / filename}; run the full experiment first.')
    metrics_df = pd.read_csv(out / 'metrics.csv')
    history_df = pd.read_csv(out / 'history.csv.gz', parse_dates=['timestamp'])
    print(f'Existing audit outputs loaded from {out}; no files overwritten.')

## <font color='#BFD72F' size=5>4.3 Quick Look</font> <a class="anchor" id="4_3"></a>

[Back to TOC](#toc)

Display the requested **metrics and per-event timings** without ranking or interpreting the results. `pred_ms_per_event` and `update_ms_per_event` come directly from `run_prequential`.

In [ ]:
display(metrics_df[[
    'city', 'representation', 'model', 'role', 'macro_f1', 'recall',
    'pred_ms_per_event', 'update_ms_per_event',
]])

#### **Insights** <a class="anchor" id='insights_2'></a>
*To be written by the group.*